In [0]:
%run ../00_setup

In [0]:
import re
from pyspark.sql.types import StructType


def flatten_structs(df, separator: str = "_"):
    """Aplana todas las columnas struct, incluyendo anidamientos de varios niveles."""
    complex_fields = {
        field.name: field.dataType
        for field in df.schema.fields
        if isinstance(field.dataType, StructType)
    }

    while complex_fields:
        col_name = list(complex_fields.keys())[0]
        expanded_columns = [
            F.col(f"{col_name}.{nested_field.name}").alias(f"{col_name}{separator}{nested_field.name}")
            for nested_field in complex_fields[col_name]
        ]
        df = df.select("*", *expanded_columns).drop(col_name)

        complex_fields = {
            field.name: field.dataType
            for field in df.schema.fields
            if isinstance(field.dataType, StructType)
        }

    return df


def to_snake_case(name: str) -> str:
    """'updatedAt' -> 'updated_at' | 'meta_updatedAt' -> 'meta_updated_at'"""
    name = name.strip()
    name = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', '_', name)
    name = name.lower()
    name = re.sub(r'[\s\.\-]+', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def standardize_column_names(df):
    """Aplica to_snake_case a TODAS las columnas del DataFrame."""
    for old_name in df.columns:
        new_name = to_snake_case(old_name)
        if new_name != old_name:
            df = df.withColumnRenamed(old_name, new_name)
    return df
